# 01: Exploratory Data Analysis (EDA)

## MovieLens Movie Recommendation Project

This notebook explores the MovieLens dataset to understand its structure, patterns, and characteristics.

### Objectives:
- Load all datasets
- Analyze data structure and types
- Explore rating distributions
- Understand user and movie patterns
- Visualize data characteristics
- Check data quality

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src to path for imports
sys.path.insert(0, str(Path.cwd()))

from src.data_loader import DataLoader

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All imports successful!")

## 2. Load Data

In [ ]:
# Load all datasets
loader = DataLoader('data')
data = loader.load_all_data()

train_df = data['train']
test_df = data['test']
movies_df = data['movies']
tags_df = data['tags']
genome_scores_df = data['genome_scores']
genome_tags_df = data['genome_tags']

## 3. Data Overview

In [ ]:
# Print dataset information
loader.get_data_info()

In [ ]:
# Check first few rows
print("Training Data Sample:")
print(train_df.head(10))

In [ ]:
print("Test Data Sample:")
print(test_df.head(10))

In [ ]:
print("Movies Sample:")
print(movies_df.head(10))

## 4. Data Quality Check

In [ ]:
# Check for missing values
print("Missing Values in Training Data:")
print(train_df.isnull().sum())
print(f"\nTotal missing: {train_df.isnull().sum().sum()}")

In [ ]:
# Data types
print("Data Types:")
print(train_df.dtypes)

## 5. Rating Distribution Analysis

In [ ]:
# Rating statistics
print("Rating Statistics:")
print(train_df['rating'].describe())
print(f"\nUnique ratings: {train_df['rating'].nunique()}")
print(f"Rating values: {sorted(train_df['rating'].unique())}")

In [ ]:
# Visualize rating distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(train_df['rating'], bins=50, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Ratings')
axes[0].grid(True, alpha=0.3)

# Count plot
rating_counts = train_df['rating'].value_counts().sort_index()
axes[1].bar(rating_counts.index, rating_counts.values, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Count')
axes[1].set_title('Count of Each Rating Value')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. User Analysis

In [ ]:
# User statistics
user_stats = train_df.groupby('userId')['rating'].agg([
    ('num_ratings', 'count'),
    ('mean_rating', 'mean'),
    ('std_rating', 'std'),
    ('min_rating', 'min'),
    ('max_rating', 'max')
]).reset_index()

print("User Statistics Summary:")
print(user_stats[['num_ratings', 'mean_rating', 'std_rating']].describe())

In [ ]:
# Most active users
print("\nTop 10 Most Active Users:")
print(user_stats.nlargest(10, 'num_ratings')[['userId', 'num_ratings', 'mean_rating']])

In [ ]:
# Visualize user activity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Number of ratings per user
axes[0].hist(user_stats['num_ratings'], bins=50, edgecolor='black', alpha=0.7, color='lightgreen')
axes[0].set_xlabel('Number of Ratings')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Ratings per User')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# Average rating by user
axes[1].hist(user_stats['mean_rating'], bins=50, edgecolor='black', alpha=0.7, color='lightyellow')
axes[1].set_xlabel('Average Rating')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Average Ratings by User')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Movie Analysis

In [ ]:
# Movie statistics
movie_stats = train_df.groupby('movieId')['rating'].agg([
    ('num_ratings', 'count'),
    ('mean_rating', 'mean'),
    ('std_rating', 'std'),
    ('min_rating', 'min'),
    ('max_rating', 'max')
]).reset_index()

# Merge with movie titles
movie_stats = movie_stats.merge(movies_df[['movieId', 'title', 'genres']], on='movieId')

print("Movie Statistics Summary:")
print(movie_stats[['num_ratings', 'mean_rating', 'std_rating']].describe())

In [ ]:
# Most popular movies
print("\nTop 15 Most Rated Movies:")
top_movies = movie_stats.nlargest(15, 'num_ratings')[['title', 'num_ratings', 'mean_rating']]
print(top_movies.to_string())

In [ ]:
# Visualize movie statistics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Number of ratings per movie
axes[0].hist(movie_stats['num_ratings'], bins=50, edgecolor='black', alpha=0.7, color='lightblue')
axes[0].set_xlabel('Number of Ratings')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Ratings per Movie')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# Average rating by movie
axes[1].hist(movie_stats['mean_rating'], bins=50, edgecolor='black', alpha=0.7, color='lightpink')
axes[1].set_xlabel('Average Rating')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Average Ratings by Movie')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Matrix Sparsity

In [ ]:
# Calculate sparsity
sparsity = loader.get_sparsity()

## 9. Genre Analysis

In [ ]:
# Parse genres (they're pipe-separated)
def parse_genres(genre_str):
    if pd.isna(genre_str):
        return []
    return genre_str.split('|')

movies_df['genre_list'] = movies_df['genres'].apply(parse_genres)

# Count genres
from collections import Counter

all_genres = []
for genres in movies_df['genre_list']:
    all_genres.extend(genres)

genre_counts = Counter(all_genres)
genre_df = pd.DataFrame(genre_counts.most_common(), columns=['Genre', 'Count'])

print("Genre Distribution:")
print(genre_df)

In [ ]:
# Visualize genres
plt.figure(figsize=(12, 6))
plt.barh(genre_df['Genre'], genre_df['Count'], color='steelblue', edgecolor='black')
plt.xlabel('Count')
plt.title('Genre Distribution in Movies')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 10. Summary and Key Insights

In [ ]:
print("="*60)
print("KEY INSIGHTS FROM EDA")
print("="*60)

print(f"\n📊 DATASET SIZE:")
print(f"  • Total ratings: {len(train_df):,}")
print(f"  • Unique users: {train_df['userId'].nunique():,}")
print(f"  • Unique movies: {train_df['movieId'].nunique():,}")
print(f"  • Test pairs: {len(test_df):,}")

print(f"\n⭐ RATINGS:")
print(f"  • Average rating: {train_df['rating'].mean():.2f}")
print(f"  • Median rating: {train_df['rating'].median():.2f}")
print(f"  • Std deviation: {train_df['rating'].std():.2f}")
print(f"  • Most common rating: {train_df['rating'].mode()[0]}")

print(f"\n👥 USER BEHAVIOR:")
print(f"  • Avg ratings per user: {user_stats['num_ratings'].mean():.1f}")
print(f"  • Median ratings per user: {user_stats['num_ratings'].median():.1f}")
print(f"  • Max ratings by one user: {user_stats['num_ratings'].max()}")
print(f"  • Min ratings by one user: {user_stats['num_ratings'].min()}")

print(f"\n🎬 MOVIE POPULARITY:")
print(f"  • Avg ratings per movie: {movie_stats['num_ratings'].mean():.1f}")
print(f"  • Median ratings per movie: {movie_stats['num_ratings'].median():.1f}")
print(f"  • Max ratings for one movie: {movie_stats['num_ratings'].max()}")
print(f"  • Min ratings for one movie: {movie_stats['num_ratings'].min()}")

print(f"\n🎯 SPARSITY:")
print(f"  • Matrix sparsity: {sparsity:.2f}%")
print(f"  • Density: {100-sparsity:.2f}%")

print("\n" + "="*60)